SETAC Europe Theme 3: Webinar registrations and member groups
Joel Aduba | Exploratory Python analysis

Analysis period: 1 October 2025 to 30 September 2026, inclusive. This is a twelve-month window ending at Busola's cut-off. It replaces the earlier September–August proposal for this notebook.
Questions to answer
1. How many identifiable Europe members had recorded membership coverage during this year? What is their breakdown by membership class, paid/free category, grade and sector?
2. Which supplied webinar files contain registrations booked within the year, and how much of the supplied collection does that represent?
3. How many eligible members registered for at least one of those webinars? How many different webinars did they register for?
4. Do recorded registration rates differ between free and paid members, and between students, recent graduates, professionals and sectors?
5. Are a small number of members registering repeatedly, or is participation spread across members?
6. How do registrations vary by month and webinar? How much of each webinar's registration list can be linked to our member population?
7. Do differences remain when we show members' differing lengths of membership coverage?
8. Which data gaps or assumptions limit the conclusions we can draw?
What these measures mean: the files do not contain an Attended field. This notebook measures recorded registrations, not verified attendance, viewing, satisfaction or benefit received. Dates refer to when bookings were made, not necessarily when webinars took place. Four supplied files contain bookings in this window; the collection is not confirmed to cover every webinar offered.

We use Busola's valid_subscriptions and person_details as the starting point. Her reconstructed coverage and historical class/grade assumptions still need checking against the SETAC Q&A. Results are exploratory until those assumptions are agreed. This notebook does not recalculate renewal outcomes.

1. Import packages and set the analysis dates
Run cells in order. The main packages are pandas, NumPy, matplotlib and seaborn; openpyxl reads Excel files. If imports fail, uncomment the installation line in your own notebook environment, run it once, then restart the kernel.

The end of the window is represented internally as 1 October 2026, exclusive, so bookings at any time on 30 September are included.

In [6]:
# If needed, run once in Jupyter, then restart the kernel:
%pip install pandas numpy matplotlib seaborn openpyxl

from pathlib import Path
from io import BytesIO
import sys
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ANALYSIS_START = pd.Timestamp('2025-10-01')
ANALYSIS_END_EXCLUSIVE = pd.Timestamp('2026-10-01')
ANALYSIS_END = ANALYSIS_END_EXCLUSIVE - pd.Timedelta(days=1)
WINDOW_DAYS = (ANALYSIS_END_EXCLUSIVE - ANALYSIS_START).days

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 85)
sns.set_theme(style='whitegrid', context='notebook')

print(f'Analysis: {ANALYSIS_START:%d %B %Y} to {ANALYSIS_END:%d %B %Y}')
print(f'Window length: {WINDOW_DAYS} days')
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 4.5 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Matplotlib is building the font cache; this may take a moment.


Analysis: 01 October 2025 to 30 September 2026
Window length: 365 days
Python: 3.13.2 | pandas: 3.0.6


2. Locate and import the relevant files
This notebook must be beside Busola's workbook and the original SETAC Europe Project Dataf file , or use your existing extracted Events/Webinars folder. The code reads the ZIP directly if no extracted webinar folder is found. It does not extract or modify source files.

The workbook can be named SETAC_Theme1_master_tables.xlsx or Copy of SETAC_Theme1_master_tables.xlsx. If both versions are present, remove the ambiguity by setting MASTER_PATH explicitly to the agreed version. If using extracted files, make sure that folder contains only the intended raw webinar exports.

Only two master sheets are loaded. We do not join webinar rows to tables A, B or C.